# 🎵 Máy chủ AI cho app Bài Hát Tặng (miễn phí)

**Cách dùng:**
1. Trên thanh menu chọn **Runtime → Change runtime type → T4 GPU → Save** (chỉ cần làm lần đầu).
2. Bấm **Runtime → Run all** (hoặc Ctrl+F9). Lần đầu mất khoảng 5–10 phút để cài và tải mô hình.
3. Kéo xuống **ô số 4 (Tạo LINK MÁY CHỦ)**. Khi dưới ô đó hiện khung xanh **LINK MÁY CHỦ** dạng `https://....trycloudflare.com`, chép link đó.
4. Mở app → ⚙️ Cài đặt → chọn **Máy chủ Colab** → dán link → Lưu → tạo bài hát.

⚠️ Giữ trang này mở trong lúc dùng app. Đóng trang hoặc để lâu không dùng thì Colab tự tắt; khi đó chạy lại và dán link mới.

Mỗi ô chạy xong sẽ hiện **khung xanh ✅ "Ô … XONG"**. Nếu hiện **khung đỏ ❌** thì làm theo hướng dẫn trong khung.

📱 **Nếu dùng điện thoại:** giữ màn hình sáng, không chuyển sang ứng dụng khác trong lúc chạy. Nếu thấy lỗi *[object CloseEvent]* hoặc mất kết nối: bấm **Kết nối lại (Reconnect)** rồi bấm ▶ chạy lại ô đang dở – việc cài đặt vẫn chạy ngầm, không phải làm lại từ đầu. Dùng máy tính sẽ ổn định hơn.

In [ ]:
#@title 1. Kiểm tra GPU
from IPython.display import HTML, display
def bao(ok, tieu_de, chi_tiet=''):
    mau, nen, icon = ('#1aa66b', '#e8f8f0', '✅') if ok else ('#d23c44', '#fdecec', '❌')
    display(HTML(f'<div style="margin-top:10px;padding:14px 16px;border:3px solid {mau};border-radius:12px;background:{nen};color:#1e3a5f;font-size:17px"><b>{icon} {tieu_de}</b><div style="font-size:15px;margin-top:6px">{chi_tiet}</div></div>'))
import subprocess
r = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    bao(True, 'Ô 1 XONG – Đã có GPU: ' + r.stdout.strip(), 'Chuyển sang ô 2.')
else:
    bao(False, 'Ô 1 LỖI – Chưa bật GPU', 'Vào menu Runtime → Change runtime type → chọn T4 GPU → Save, rồi bấm Runtime → Run all lại.')


In [ ]:
#@title 2. Cài ACE-Step 1.5 (3–8 phút, chạy ngầm – mất kết nối thì bấm chạy lại ô này)
from IPython.display import HTML, display
def bao(ok, tieu_de, chi_tiet=''):
    mau, nen, icon = ('#1aa66b', '#e8f8f0', '✅') if ok else ('#d23c44', '#fdecec', '❌')
    display(HTML(f'<div style="margin-top:10px;padding:14px 16px;border:3px solid {mau};border-radius:12px;background:{nen};color:#1e3a5f;font-size:17px"><b>{icon} {tieu_de}</b><div style="font-size:15px;margin-top:6px">{chi_tiet}</div></div>'))
import os, subprocess, time
DONE = '/content/ace_cai_xong.txt'
LOG = '/content/cai_dat.log'
script = r"""
set -e
cd /content
[ -f /content/ace/pyproject.toml ] || git clone -q --depth 1 https://github.com/ace-step/ACE-Step-1.5 /content/ace
pip -q install uv
cd /content/ace
uv sync
echo OK > /content/ace_cai_xong.txt
"""
PID = '/content/cai_dat.pid'
def dang_chay():
    try:
        pid = int(open(PID).read().strip()); os.kill(pid, 0)
        st = open('/proc/%d/stat' % pid).read().split()[2]
        return st != 'Z'
    except Exception:
        return False
if os.path.exists(DONE):
    bao(True, 'Ô 2 XONG – ACE-Step đã được cài từ trước', 'Chuyển sang ô 3.')
else:
    if not dang_chay():
        open('/content/cai_dat.sh', 'w').write(script)
        pr = subprocess.Popen(['bash', '/content/cai_dat.sh'], stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, start_new_session=True)
        open(PID, 'w').write(str(pr.pid))
        print('🚀 Bắt đầu cài đặt (chạy ngầm trên máy Colab)...')
    else:
        print('⏳ Việc cài đặt vẫn đang chạy từ lần trước, tiếp tục đợi...')
    t0 = time.time()
    while not os.path.exists(DONE):
        time.sleep(15)
        if not dang_chay() and not os.path.exists(DONE):
            break
        tail = subprocess.run('tail -c 300 ' + LOG, shell=True, capture_output=True, text=True).stdout.strip().splitlines()
        print('⏳ Đang cài... %d giây | %s' % (time.time() - t0, (tail[-1] if tail else '')[:120]), flush=True)
    if os.path.exists(DONE):
        bao(True, 'Ô 2 XONG – Đã cài ACE-Step', 'Chuyển sang ô 3.')
    else:
        print(open(LOG).read()[-2500:])
        bao(False, 'Ô 2 LỖI – Cài đặt không thành công', 'Bấm Runtime → Disconnect and delete runtime, rồi Run all lại. Nếu vẫn lỗi, chụp màn hình phần chữ phía trên gửi lại.')


In [ ]:
#@title 3. Khởi động máy chủ (lần đầu tải mô hình khoảng 3–5 phút)
from IPython.display import HTML, display
def bao(ok, tieu_de, chi_tiet=''):
    mau, nen, icon = ('#1aa66b', '#e8f8f0', '✅') if ok else ('#d23c44', '#fdecec', '❌')
    display(HTML(f'<div style="margin-top:10px;padding:14px 16px;border:3px solid {mau};border-radius:12px;background:{nen};color:#1e3a5f;font-size:17px"><b>{icon} {tieu_de}</b><div style="font-size:15px;margin-top:6px">{chi_tiet}</div></div>'))
wrapper = r'''
import os, sys
os.environ['MPLBACKEND'] = 'Agg'   # Colab đặt backend riêng, máy chủ không dùng được
import acestep.api.route_setup as rs
from fastapi.middleware.cors import CORSMiddleware as _C
class OpenCORS(_C):
    def __init__(self, app, **kw):
        super().__init__(app, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
rs.CORSMiddleware = OpenCORS   # cho phép app điện thoại gọi tới
from acestep.api_server import main
sys.argv = ['acestep-api', '--host', '0.0.0.0', '--port', '8001']
main()
'''
open('/content/ace/run_server.py','w').write(wrapper)
import subprocess, time, urllib.request
subprocess.run('pkill -f run_server.py', shell=True)
log = open('/content/server.log','w')
import os
env = dict(os.environ, MPLBACKEND='Agg')
srv = subprocess.Popen(['uv','run','--no-sync','python','run_server.py'], cwd='/content/ace', stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
ok = False
for i in range(240):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=3); ok = True; break
    except Exception:
        if srv.poll() is not None: break
        if i % 6 == 0: print('⏳ Đang khởi động...', i*5, 'giây')
if ok: bao(True, 'Ô 3 XONG – Máy chủ AI đã chạy', 'Chuyển sang ô 4 để lấy link.')
else:
    print(open('/content/server.log').read()[-3000:])
    bao(False, 'Ô 3 LỖI – Máy chủ không khởi động được', 'Xem nhật ký phía trên. Thử Runtime → Run all lại; nếu vẫn lỗi, chụp màn hình gửi lại.')

In [ ]:
#@title 4. Tạo LINK MÁY CHỦ để dán vào app
from IPython.display import HTML, display
def bao(ok, tieu_de, chi_tiet=''):
    mau, nen, icon = ('#1aa66b', '#e8f8f0', '✅') if ok else ('#d23c44', '#fdecec', '❌')
    display(HTML(f'<div style="margin-top:10px;padding:14px 16px;border:3px solid {mau};border-radius:12px;background:{nen};color:#1e3a5f;font-size:17px"><b>{icon} {tieu_de}</b><div style="font-size:15px;margin-top:6px">{chi_tiet}</div></div>'))
import subprocess, re, time, os
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
subprocess.run('pkill -f cloudflared', shell=True)
tl = open('/content/tunnel.log','w')
tun = subprocess.Popen(['/content/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8001'], stdout=tl, stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
from IPython.display import HTML, display
if url:
    print('\n' + '='*60)
    print('LINK MÁY CHỦ (chép dòng dưới đây):')
    print(url)
    print('='*60 + '\n')
    display(HTML(f"""<div style="font-size:18px;padding:16px;border:3px solid #0a78bd;border-radius:12px;background:#eef6fd;color:#1e3a5f">
    <b style="color:#1aa66b">✅ Ô 4 XONG – Đã có link máy chủ</b><br>🔗 <b>LINK MÁY CHỦ:</b><br>
    <input id="lk" value="{url}" readonly onclick="this.select()" style="width:100%;font-size:18px;padding:10px;margin:10px 0;border:2px solid #0a78bd;border-radius:8px">
    <button onclick="navigator.clipboard.writeText('{url}').then(()=>this.innerText='✅ Đã chép! Dán vào app')" style="font-size:18px;padding:12px 18px;background:#0a78bd;color:#fff;border:0;border-radius:10px">📋 Chép link</button>
    <p>Dán vào app → ⚙️ Cài đặt → Máy chủ Colab → Lưu. Giữ trang này mở khi dùng app.</p></div>"""))
else:
    print(open('/content/tunnel.log').read()[-1500:])
    bao(False, 'Ô 4 LỖI – Chưa tạo được link', 'Bấm nút ▶ bên trái ô này để chạy lại ô 4.')


In [ ]:
#@title 5. (Tùy chọn) Xem nhật ký máy chủ
from IPython.display import HTML, display
def bao(ok, tieu_de, chi_tiet=''):
    mau, nen, icon = ('#1aa66b', '#e8f8f0', '✅') if ok else ('#d23c44', '#fdecec', '❌')
    display(HTML(f'<div style="margin-top:10px;padding:14px 16px;border:3px solid {mau};border-radius:12px;background:{nen};color:#1e3a5f;font-size:17px"><b>{icon} {tieu_de}</b><div style="font-size:15px;margin-top:6px">{chi_tiet}</div></div>'))
import subprocess
print(subprocess.run('tail -n 40 /content/server.log', shell=True, capture_output=True, text=True).stdout)
bao(True, 'Ô 5 XONG – Đã hiện nhật ký máy chủ', 'Chỉ dùng khi cần xem lỗi. Giữ trang mở trong lúc dùng app.')
